<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## ✂️ 长对话管理机制全表

**一句话定位**:messages 越长越贵、越慢、越乱(context rot)——**少发、发对、别重复付费**。
> 标 ✅ 的行为已在本机 langgraph 1.2 / langchain 1.3 / langchain-core 1.4 实测。

</div>

### 1️⃣ 全景:按"动哪里"分类

| 层 | 机制 | 改 state? | 省钱 | 提质 |
|---|---|---|---|---|
| **A 视图** | 切片 · `trim_messages` · `filter_messages` · `merge_message_runs` | ❌ 只改发给 LLM 的视图 | ✅ | ✅ |
| **B 删除** | `RemoveMessage` | ✅ 永久删 | ✅ | ✅ |
| **C 压缩** | 滚动摘要 · `SummarizationMiddleware` | ✅ 旧消息→摘要 | ✅ | ⚠️ 有损 |
| **D 工具结果** | `ContextEditingMiddleware` · 截断 / 落盘 / `artifact` | 视情况 | ✅✅ | ✅ |
| **E 工具定义** | `LLMToolSelectorMiddleware` | ❌ | ✅ | ✅ |
| **F 外置记忆** | Store + 检索 · RAG | ❌ | ✅✅ | ✅ |
| **G 架构** | subagent 隔离 context | ❌ | ✅✅ | ✅✅ |
| **H 计费** | prompt caching · 便宜模型 · 调用上限 | ❌ | ✅✅ | — |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅰️ 视图层:不动 state,只改喂给 LLM 的内容(course 已讲)

</div>

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

llm = init_chat_model("openai:gpt-4o")
state = {"messages": [HumanMessage("hi")]}
msgs = state["messages"]

from langchain_core.messages import trim_messages, filter_messages, merge_message_runs

llm.invoke(state["messages"][-1:])                          # ① 切片:最简单

trim_messages(msgs, max_tokens=1000, token_counter="approximate",   # ② 按预算裁剪
              strategy="last", include_system=True, start_on="human")

filter_messages(msgs, include_types=["human", "ai"])         # ③ 按条件过滤
filter_messages(msgs, exclude_tool_calls=["call_id_1"])      #    剔除指定 tool call(+其 ToolMessage)
merge_message_runs(msgs)                                     # ④ 合并连续同角色消息

| API | 关键参数 | 实测要点 ✅ |
|---|---|---|
| 切片 `[-n:]` | — | 最快;可能切断 tool_call ↔ ToolMessage 配对 |
| `trim_messages` | `max_tokens` · `token_counter`(`len` 按条 / `"approximate"` / 模型) · `strategy`(`last`/`first`) · `include_system` · `start_on` · `end_on` · `allow_partial` | `include_system=True` 保留 system;`start_on="human"` 保证以 human 开头;**预算太小可能返回 `[]`** |
| `filter_messages` | `include_/exclude_` × `types` · `names` · `ids`;`exclude_tool_calls`(**传 tool_call id 列表**或 `True`) | `exclude_tool_calls` 会连带删对应 `ToolMessage` |
| `merge_message_runs` | — | 合并连续同角色消息 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:视图层**不改 state**——实测节点里 `trim_messages` 后 state 仍是 7 条,只是 LLM 只看到 2 条。checkpoint 仍存全量;要真删请用 B 层。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅱️ 删除层:改 state

</div>

In [ ]:
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES

def prune(state):
    return {"messages": [RemoveMessage(id=m.id) for m in state["messages"][:-2]]}   # 只留最近 2 条

def wipe(state):
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES)]}                    # 清空全部

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

依赖 `add_messages` reducer;`RemoveMessage` 的 id 不存在会 `ValueError`。**删 AI 的 tool_call 消息时要连带删它的 ToolMessage**,否则 provider 报错。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅲 压缩层:有损但保留语义

</div>

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.graph import MessagesState

llm = init_chat_model("openai:gpt-4o")

# ① 滚动摘要(course 005 手写模式):state 加 summary 字段
class State(MessagesState):
    summary: str

def summarize(state: State):
    prompt = f"已有摘要:{state.get('summary','')}\n请把以上对话并入摘要" if state.get("summary") else "总结以上对话"
    new = llm.invoke(state["messages"] + [HumanMessage(prompt)])
    delete = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]    # 保留最近 2 条
    return {"summary": new.content, "messages": delete}
# 调用模型前:把 summary 作为 SystemMessage 拼回去

# ② 预构建中间件(create_agent)
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    "openai:gpt-4o", tools=[],
    middleware=[SummarizationMiddleware(
        model="openai:gpt-4o-mini",          # 摘要用便宜模型
        trigger=("tokens", 4000),            # 触发:tokens / messages / fraction,可传 list
        keep=("messages", 20),               # 保留最近多少
    )],
)

| 项 | 内容 |
|---|---|
| `trigger` | `("tokens", n)` · `("messages", n)` · `("fraction", 0.8)` 或它们的 list ✅签名 |
| `keep` | 默认 `("messages", 20)`;同三种单位 ✅签名 |
| 代价 | 额外一次 LLM 调用 + 细节丢失 → 摘要模型用小模型,关键事实另存 Store |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅲➕ 滚动摘要完整流程(course 005)

</div>

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("openai:gpt-4o")
checkpointer = InMemorySaver()

from typing_extensions import Literal
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage
from langgraph.graph import StateGraph, START, END, MessagesState

class State(MessagesState):
    summary: str                                   # 在 messages 之外加一个 summary key

def call_model(state: State):                      # 主对话节点:有摘要就拼成 SystemMessage
    summary = state.get("summary", "")
    msgs = ([SystemMessage(f"Summary of conversation earlier: {summary}")] + state["messages"]
            if summary else state["messages"])
    return {"messages": model.invoke(msgs)}

def summarize_conversation(state: State):          # 摘要节点:生成/延伸摘要 → 删旧消息
    summary = state.get("summary", "")
    prompt = (f"This is summary of the conversation to date: {summary}\n\n"
              "Extend the summary by taking into account the new messages above:"
              if summary else "Create a summary of the conversation above:")
    new = model.invoke(state["messages"] + [HumanMessage(prompt)])
    delete = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]      # 只留最近 2 条
    return {"summary": new.content, "messages": delete}

def should_continue(state: State) -> Literal["summarize_conversation", END]:
    return "summarize_conversation" if len(state["messages"]) > 6 else END  # 触发阈值

workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)
graph = workflow.compile(checkpointer=checkpointer)     # 摘要要跨轮保留 → 必须有 checkpointer

```
START → conversation ──消息数 > 6──▶ summarize_conversation → END
                     └──否则────────▶ END
```

| 环节 | 要点 |
|---|---|
| 摘要**放哪** | state 的 `summary` key,**不是**消息;下次调用时拼进 `SystemMessage` |
| 摘要**何时生成** | 每轮回答**之后**(`conversation` 之后判断),不是回答之前 |
| 摘要**怎么延伸** | 已有摘要 → prompt 要求"在其基础上延伸";无则新建 |
| 删什么 | `RemoveMessage` 删到只剩最近 2 条 |
| 实测(假模型)✅ | 每轮追加 2 条:2 → 4 → 6 → **第 4 轮达 8 条 > 6 触发**,摘要生成,messages 缩成 2;之后继续追加 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:摘要在**下一轮**才生效——本轮触发摘要时,本轮回答已经用完整历史生成了;被删掉的历史只能靠 `summary` 保留语义(有损)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅲➕ 持久化到外部 DB(course 006)

</div>

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver     # pip: langgraph-checkpoint-sqlite

conn = sqlite3.connect("state_db/example.db", check_same_thread=False)   # 文件路径 = 真持久化
# conn = sqlite3.connect(":memory:", check_same_thread=False)            # 内存库:效果≈InMemorySaver
checkpointer = SqliteSaver(conn)

graph = workflow.compile(checkpointer=checkpointer)     # 其余代码与摘要版完全一致
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": [HumanMessage("hi! I'm Lance")]}, config)
graph.get_state(config)                                 # kernel 重启后用同 db + thread_id 仍能取回

| 项 | 内容 |
|---|---|
| 与 `InMemorySaver` 区别 | state 写到磁盘,**进程 / kernel 重启后仍在** |
| 无需改图 | 只换 `compile(checkpointer=...)` 的实例 |
| 实测 ✅ | `SqliteSaver` + 新建连接重开同一 db:`messages=2`、`summary` 仍在,继续对话正常(摘要版图) |
| `check_same_thread=False` | LangGraph 多线程访问需要,必须加 |
| 其他库 | `PostgresSaver`(生产) 等,换 saver 即可 |

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

本地调试:在 `module-2/studio` 下跑 `langgraph dev`,用 **LangSmith Studio**(原 LangGraph Studio)浏览器 UI 加载 `chatbot`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅳 工具结果层:长对话的头号膨胀源

</div>

In [ ]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

ContextEditingMiddleware(edits=[ClearToolUsesEdit(
    trigger=100_000,          # 超过此 token 才清
    keep=3,                   # 保留最近 3 个工具结果
    clear_tool_inputs=False,  # 是否连工具入参一起清
    exclude_tools=(),         # 不清的工具名
    placeholder="[cleared]",  # 替换文本
)])

| 手段 | 做法 |
|---|---|
| 清旧工具结果 | 上面的 `ClearToolUsesEdit`(参数 ✅签名) |
| 返回时截断 | 工具内只返回摘要 / 前 N 字 |
| 落盘 | 大结果写文件,只回传路径(deep-agents 的 Files 治挤) |
| `ToolMessage.artifact` | 原始数据留给下游代码,**不发给模型** |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅴🅵🅶 工具定义 · 外置记忆 · 架构

</div>

| 机制 | 用法 | 解决 |
|---|---|---|
| **E 工具筛选** | `LLMToolSelectorMiddleware(model=…, max_tools=3, always_include=[…])` ✅签名 | 工具太多,每次只发相关的定义 |
| **F Store 检索** | 事实存 `store.put`,用时 `store.search` 只取相关几条(见 04 笔记) | 不让全部历史常驻 context |
| **F RAG** | 外部文档向量检索,按需注入 | 知识大于窗口 |
| **G subagent 隔离** | 子任务在独立 context 跑,只回传结论 | 中间过程不污染主 context |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🅷 计费层

</div>

| 机制 | 要点 |
|---|---|
| **Prompt caching** | 稳定的**前缀**(system + 工具 + 早期消息)命中缓存更便宜;⚠️ **从头部裁剪 / 改写早期消息会破坏缓存**——倾向"只追加 + 到阈值再一次性压缩" |
| 便宜模型 | 摘要 / 筛选 / 分类用 mini 级 |
| 上限保险 | `ModelCallLimitMiddleware` · `ToolCallLimitMiddleware` ✅存在:防循环烧钱 |
| 观测 | LangSmith 看每步 token,先找最胖的那条消息再优化 |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧭 怎么选

</div>

| 症状 | 首选 |
|---|---|
| 对话轮数多 | 滚动摘要(`summary` key + `RemoveMessage`)/ `SummarizationMiddleware` |
| 重启后记忆不能丢 | `SqliteSaver` / `PostgresSaver` 代替 `InMemorySaver` |
| 只要最近几轮 | `trim_messages`(`start_on="human"`) |
| 工具返回巨大 | `ClearToolUsesEdit` + 截断 / 落盘 |
| 工具太多 | `LLMToolSelectorMiddleware` |
| 要跨会话记住事实 | Store(长期记忆) |
| 任务链长、中间产物多 | subagent 隔离 |
| 账单高 | 先开 prompt caching,再压缩 |

**组合套路**:system 固定 → 旧工具结果清除 → 超阈值摘要 → 视图层 `trim` 兜底。

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**三大坑**:① `trim`/切片切断 `tool_call ↔ ToolMessage` 配对 → provider 400 ② `trim` 预算太小返回 `[]` ③ 视图层不改 state,checkpoint 持续变胖。

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**:**视图层(trim/filter)最轻、删除层(RemoveMessage)真省 state、压缩层(摘要)保语义、工具结果先瘦身、重活交给 Store 与 subagent、最后用 prompt caching 压账单。**

</div>